# Speculative Decoding Hands-On Lab
### Section 2.4 of *Speculative Decoding: What Lossless Means, What It Doesn't, and What's Next*

This notebook walks the lab end to end. Every output cell below is a **real measurement** from our run (one H100, September 2026), so you can read the whole experiment without spending GPU time. To reproduce, run the commands with your own Modal account: an H100 is ~$4/hour and the full lab uses $8–12 (Modal gives $30 free credits on signup).

**Engine map** (verified 2026-09-02):

| DeepSpec checkpoint | SGLang | vLLM |
|---|---|---|
| `dspark_qwen3_8b_block7` | ✅ | ✅ |
| `eagle3_qwen3_8b_ttt7` | ❌ | ✅ `qwen3_eagle3` |
| `dflash_qwen3_8b_block7` | ❌ (`markov_rank=0`) | ✅ offline path, arch relabel |

Stages 1–2 run on vLLM; Stage 3 runs on SGLang, because only SGLang exposes the acceptance threshold. That asymmetry is itself Section 2.2's lesson.

> **Cost:** after each stage, release the GPU with `modal app stop -y <app>` and confirm the URL returns 404. A silent no-op keeps billing.

## Stage 1 — Serve your first accelerated model

Serve the same Qwen3-8B twice, once vanilla and once with the DSpark draft, and measure the speedup on your own GPU. First deployment cold-starts ~10 minutes (image build + 16GB weights), cached afterwards.

In [ ]:
!pip install modal && modal setup                      # one-time account link
!SPEC_MODE=vanilla modal deploy modal_vllm_serve.py     # prints your server URL
!python3 measure_decoding_speed.py --url $URL --label vanilla
!SPEC_MODE=dspark modal deploy modal_vllm_serve.py
!python3 measure_decoding_speed.py --url $URL --label dspark

[vanilla] 136.3 ± 1.3 tok/s   (mean ± std over 5 runs)
[dspark]  231.4 ± 4.5 tok/s   → 1.70x speedup


Where does the speedup come from? vLLM does not report acceptance length directly; both numbers fall out of the `/metrics` counters (5,180 draft tokens proposed at 7 per pass ≈ 740 verification passes for 2,606 generated tokens):

```text
L_target = 1 / 136.3 tok/s ≈ 7.3 ms      # per-token latency, vanilla
L_dspark = 1 / 231.4 tok/s ≈ 4.3 ms      # per-token latency, with draft
τ        ≈ 3.5                           # acceptance length
η = L_target / L_dspark ≈ 1.70x          # speedup
```

![Decode throughput, vanilla vs DSpark on one H100](../interactive_site/figures/fig13_runs_h100.svg)

*Decode throughput of Qwen3-8B on one H100, vanilla vs the DSpark draft. Mean ± std over 5 runs.*

## Stage 2 — The decoding race

Four deployments race on the same prompt: vanilla vs EAGLE-3 vs DFlash vs DSpark. One prompt per domain, 512 tokens, greedy, median of 5 runs — a probe of domain-conditional acceptance, not a benchmark.

In [ ]:
!SPEC_MODE=eagle3 modal deploy modal_vllm_serve.py
!modal run modal_dflash_offline.py        # DFlash lane: vllm serve crashes on it
!python3 race_domains.py --url $URL

domain    vanilla   DSpark          EAGLE-3         DFlash
coding    138.1     311.9 (2.3x)    158.8 (1.15x)   311.3 (2.25x)
creative  138.2     416.7 (3.0x)    229.5 (1.66x)   265.6 (1.92x)
frontend  137.6     333.1 (2.4x)    208.2 (1.51x)   274.0 (1.99x)


**Question before you scroll on:** why does the same draft buy 3.0x on creative but 2.3x on coding?

Because acceptance is domain-conditional: τ measures how often the draft guesses what the target would say, and that agreement varies with the text. Vanilla is flat across domains; the speculators are not. (Caveat: at temperature 0, open-ended prose loops, and repetitive text is easy to draft — rerun at temperature 0.7 and compare.)

![Race demo, frontend brief](../interactive_site/figures/fig16_race_demo_frontend.jpg)

*The decoding race on the LosslessBench calendar brief (L101): vanilla 18.7s, DFlash 8.9s.*

![Race demo, creative brief](../interactive_site/figures/fig17_race_demo_creative.jpg)

*The same race on a 1000-word creative brief (L073): vanilla 15.2s, DFlash 8.5s.*

## Stage 3 — Adjust the acceptance threshold (the Figure 14 demo)

SGLang ships `--speculative-accept-threshold-single` at 1.0 (strict rejection sampling, lossless). This sweep redeploys the server per threshold, regenerates the LosslessBench L101 frontend page (temperature 1, 3 samples per stop), and records acceptance length and speed.

In [ ]:
!python3 sweep_frontend_knob.py --url https://<you>--neurips-lab-sglang-serve.modal.run \
    --temperature 1 --samples 3 --seed 42

threshold  1.0    0.8    0.6    0.4
τ          4.88   5.85   6.68   7.23    (+48% from strict to 0.4)
tok/s      470    552    546    560
pages      th_1.0_s*.html ... th_0.4_s*.html  (open side by side)


τ climbs 48% as the threshold relaxes: draft-preferred tokens are accepted without resampling and the output distribution shifts toward the draft. Open the generated pages side by side and judge the quality with your own eyes — acceptance rate is not accuracy.

**Teaching point:** at temperature 0 the threshold is inert. We swept the same range greedy and every page came back byte-identical, because a draft token is only ever accepted when it already matches the argmax. Verify it yourself with `--temperature 0`.

## Stage 4 — Cross-domain evaluation with LosslessBench

The task manifests and hydrator live in `../losslessbench/` (full data: [huggingface.co/datasets/lilyzhng/lossless100](https://huggingface.co/datasets/lilyzhng/lossless100)). Generate the same brief across lanes and compare outputs per domain:

In [ ]:
!python3 generate_frontend_task.py --url $URL --label vanilla   # rerun per lane
!python3 generate_creative_task.py --url $URL --label vanilla

vanilla vs dspark (greedy, L101): identical for 8,299 chars (76% of the page),
then diverges at one CSS value (transition: color 0.2s -> 0.3s),
and the trajectories separate from there (10,924 vs 10,535 chars).


![Five-domain radar](../interactive_site/figures/fig_radar_spec_pilot.svg)

*Qwen3-8B with vs without speculative decoding across the five LosslessBench domains (Figure 11 of the article). Axes are independently scaled, so each domain's relative gap is visible.*

Not the theorem failing: the guarantee is about distributions, not trajectories. The speculative path runs different kernels, numerics shift by a hair, and a near-tie token falls the other way. If your product depends on reproducing an exact output, lossless-in-distribution is not the property you think it is.

## Wrap up

```bash
modal app stop -y neurips-spec-lab
modal app stop -y neurips-lab-sglang     # then confirm the URLs return 404
```

Three things to take away: what τ measures (draft–target agreement per verification pass), why acceptance is domain-conditional, and why acceptance rate is not accuracy.